In [1]:
from os import sched_get_priority_max

from pykeen.triples import TriplesFactory
import pandas as pd
from rdflib import Graph, Namespace, URIRef, BNode
from rdflib.namespace import RDF, RDFS, OWL
from rdflib.collection import Collection


import csv
from tqdm import tqdm # A library for a smart progress bar
from collections import defaultdict
import sys
from collections import Counter

from pykeen import datasets
from pykeen.triples import TriplesFactory

/Users/anon/miniconda3/envs/calibration/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# YAGO 4.5-10

In [2]:
# generates the base yago4.5-10 by counting and filtering

def filter_by_entity_count(file_path, output_file, threshold=10):
    """
    Filters an n-triples file to keep lines where both the subject
    and object entities appear more than a given number of times.

    Args:
        file_path (str): The path to the n-triples file.
        threshold (int): The minimum count for an entity to be included.
    """
    entity_counts = Counter()

    # --- First Pass: Count all entities ---
    # This pass builds a frequency map of every subject and object.
    print(f"INFO: Starting first pass to count entities from '{file_path}'...")
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 3:
                    subject = parts[0]
                    obj = parts[2]
                    entity_counts[subject] += 1
                    entity_counts[obj] += 1
    except FileNotFoundError:
        print(f"ERROR: File not found at '{file_path}'", file=sys.stderr)
        return
    except Exception as e:
        print(f"ERROR: An error occurred during the first pass: {e}", file=sys.stderr)
        return

    print(f"INFO: First pass complete. Found {len(entity_counts)} unique entities.")

    # --- Second Pass: Filter and print ---
    # This pass re-reads the file and prints lines that meet the criteria.
    print(f"INFO: Starting second pass to filter lines with entity counts > {threshold}...")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            with open(output_file, 'w', encoding='utf-8') as of:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 3:
                        subject = parts[0]
                        obj = parts[2]
                        # Check if BOTH entities are above the threshold
                        if entity_counts[subject] > threshold and entity_counts[obj] > threshold:
                            # Print the original, unmodified line to standard output
                            of.write(line)
    except Exception as e:
        print(f"ERROR: An error occurred during the second pass: {e}", file=sys.stderr)
        return

    print("INFO: Filtering complete.")

input_file = 'YAGO4.5/data/original/yago4.5_triples.nt'
output_file = 'YAGO4.5/data/original/yago4.5-10.nt'
filter_by_entity_count(input_file, output_file)

INFO: Starting first pass to count entities from 'YAGO4.5/data/original/yago4.5_triples.nt'...
INFO: First pass complete. Found 4686833 unique entities.
INFO: Starting second pass to filter lines with entity counts > 10...
INFO: Filtering complete.


In [3]:
#extracts all rdf:type triples and removes trivial 'class'
!grep '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>' YAGO4.5/data/original/yago-facts-nolit.nt | grep -v ' <http://www.w3.org/2000/01/rdf-schema#Class>' > YAGO4.5/data/original/YAGO4.5_entity_types.nt



In [4]:
# filters to get only yago4.5-10 entities and their direct classes
with open('YAGO4.5/data/original/yago4.5-10.nt') as fp:
    entities = set()
    for line in fp.readlines():
        e1, p, e2, _ = line.split()
        entities.add(e1)
        entities.add(e2)

with open('YAGO4.5/data/original/YAGO4.5_entity_types.nt') as inFile:
    with open('YAGO4.5/data/YAGO4.5-10_entity_types.nt', 'w') as outFile:
        fullcnt = 0
        cnt = 0
        for line in inFile.readlines():
            fullcnt+=1
            ent, *rest = line.split()
            if ent in entities:
                outFile.write(line)
                cnt += 1

this was a different file

In [36]:
#first do sed 's/ \.$//' yago4.5-10.nt > yago4.5-10.txt
tf = TriplesFactory.from_path('YAGO4.5/data/original/yago4.5-10.txt', create_inverse_triples=False, load_triples_kwargs={'delimiter': ' '})
training, testing, validation = tf.split([.99, .005, .005],random_state=42)
print(training.num_triples)
print(testing.num_triples)
print(validation.num_triples)

3222052
16273
16273


In [27]:
e_conversion_dict = {value: key for key, value in training.entity_to_id.items()}
r_conversion_dict = {value: key for key, value in training.relation_to_id.items()}

Extract schema used in our model

In [38]:
for output_filename, input_triples in [('YAGO4.5/data/YAGO4.5-10_train.tsv',training.mapped_triples),
                                       ('YAGO4.5/data/YAGO4.5-10_valid.tsv',validation.mapped_triples),
                                       ('YAGO4.5/data/YAGO4.5-10_test.tsv',testing.mapped_triples)]:
    batch_size = 100_000
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
    # Create a CSV writer with a tab delimiter
        writer = csv.writer(f, delimiter='\t')

        # Use tqdm for a helpful progress bar
        # We iterate through the tensor in steps of batch_size
        for i in tqdm(range(0, input_triples.shape[0], batch_size)):
            # Get a chunk of the tensor
            chunk = input_triples[i : i + batch_size]

            # Move chunk to CPU (if it's on GPU) and convert to a Python list
            # .tolist() is efficient for converting a small chunk
            chunk_list = chunk.cpu().tolist()

            # Map the integer values to strings using the dictionary.
            # We use .get() for safety in case a key is missing.
            mapped_rows = [
                [e_conversion_dict.get(row[0], 'KEY_NOT_FOUND'),
                 r_conversion_dict.get(row[1], 'KEY_NOT_FOUND'),
                 e_conversion_dict.get(row[2], 'KEY_NOT_FOUND')]
                for row in chunk_list
            ]

            # Write the mapped rows to the TSV file
            writer.writerows(mapped_rows)

100%|██████████| 1/1 [00:00<00:00, 21.80it/s]


In [6]:
schemaorg = Graph()
schemaorg.parse('YAGO4.5/data/original/00-schema-org.ttl',format='turtle')#this is the schema from schema.org
properties = pd.read_csv('YAGO4.5/data/YAGO4.5-10_train.tsv', sep='\t', header=None, names=['s','p','o'])['p'].unique()

In [29]:
domains= defaultdict(list)
ranges= defaultdict(list)

def make_query(name, domain=True):
    if domain:
        q = '''PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        prefix schema: <http://schema.org/>
        select ?result
        where {{
        {property} schema:domainIncludes ?result .
        }}
    '''.format(property=name)
    else :
        q = '''PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        prefix schema: <http://schema.org/>
        select ?result
        where {{
        {property} schema:rangeIncludes ?result .
        }}
    '''.format(property=name)
    return q

for prop in properties:
    q= make_query(prop)

    res= [str(res.result) for res in schemaorg.query(q)]
    domains[prop] = res

    q= make_query(prop,False)

    res= [str(res.result) for res in schemaorg.query(q)]
    ranges[prop] = res


In [40]:
pd.DataFrame.from_records([domains,ranges]).transpose().to_csv('yago_properties.csv', index=True, header=False)

### from the 'handcrafted file' create a rdflib graph to deal with unions of domain/ranges

In [14]:
drf = pd.read_csv('YAGO4.5/data/original/properties_d_r_f.csv', header=None, names=['prop','domains','ranges','functional'])
drf.fillna(False, inplace=True)

/var/folders/ct/yy2gltkx6n56mgt7r9gf4qsm0000gn/T/ipykernel_19664/3372458310.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  drf.fillna(False, inplace=True)


In [15]:
tboxgraph = Graph()
for i,row in drf.iterrows():
    prop = URIRef(row.prop.strip('<').strip('>'))
    domains = [URIRef(dom.strip('<').strip('>')) for dom in eval(row.domains)] #this is stupid but necessary
    ranges = [URIRef(ran.strip('<').strip('>')) for ran in eval(row.ranges)]
    #functional = row.functional
    tboxgraph.add((prop, RDF.type, RDF.Property))
    if row.functional:
        tboxgraph.add((prop, RDF.type, OWL.FunctionalProperty))
    if len(domains) == 1:

        tboxgraph.add((prop, RDFS.domain, domains[0]))
    else:
        # Complex case: create a union class for the domain
        union_bnode = BNode() # An anonymous class for the union
        tboxgraph.add((union_bnode, RDF.type, RDFS.Class))
        # Use rdflib's Collection to create the RDF list for owl:unionOf
        c = Collection(tboxgraph, BNode(), domains)
        tboxgraph.add((union_bnode, OWL.unionOf, c.uri))
        tboxgraph.add((prop, RDFS.domain, union_bnode))

    if len(ranges) == 1:

        tboxgraph.add((prop, RDFS.range, ranges[0]))
    else:
        # Complex case: create a union class for the domain
        union_bnode = BNode() # An anonymous class for the union
        tboxgraph.add((union_bnode, RDF.type, RDFS.Class))
        # Use rdflib's Collection to create the RDF list for owl:unionOf
        c = Collection(tboxgraph, BNode(), ranges)
        tboxgraph.add((union_bnode, OWL.unionOf, c.uri))
        tboxgraph.add((prop, RDFS.range, union_bnode))
print(len(tboxgraph))
tboxgraph.parse('YAGO4.5/data/original/yago-taxonomy.ttl', format='ttl')
print(len(tboxgraph))
tboxgraph.parse('YAGO4.5/data/original/yago-disjoints.ttl', format='ttl')
tboxgraph.serialize('../scratchbook/YAGO4.5-10_tbox_only.nt', format='nt')
print(len(tboxgraph))
tboxgraph.parse('YAGO4.5/data/YAGO4.5-10_entity_types.nt', format='nt')
print(len(tboxgraph))
tboxgraph.serialize('YAGO4.5/data/YAGO4.5-10_tbox.nt', format='nt')
tboxgraph.parse('YAGO4.5/data/original/yago4.5-10.nt', format='nt')
print(len(tboxgraph))
tboxgraph.serialize('YAGO4.5/data/YAGO4.5-10_full_graph.nt', format='nt')


360
166720


/Users/anon/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


166776
742447
3997045


<Graph identifier=N754b49f9a25c4011aa29a078191dfca5 (<class 'rdflib.graph.Graph'>)>

# CSKG

In [2]:
problems = ['https://w3id.org/cskg/ontology#'+ prob for prob in ['skos:broader','matchesOtherEntity','matchesMaterial','annotatesMetric','executesMetric']] #do not exist in current schema
path = 'CSKG2/data/original/'
datapath= 'CSKG2/data/'

In [28]:

i2e = list()
i2r = list()


with open(path+'entities.txt') as ent:
    for line in ent.readlines():
        value, name = line.split()
        i2e.append('https://w3id.org/cskg/ontology#'+name)

with open(path+'relations.txt') as rel:
    for line in rel.readlines():
        value, name = line.split()
        if 'skos' in name:
            i2r.append(name.replace('skos:','http://www.w3.org/2004/02/skos/core#'))
        else:
            i2r.append('https://w3id.org/cskg/ontology#'+name)


cnt = 0
tot = 0
for fname in ['train','valid','test']:
    with open(path+fname+'.txt') as inFile:
        with open(datapath+fname+'.tsv','w') as outFile:
            for line in inFile:
                tot += 1
                s, p, o = line.split()
                # if i2r[int(p)] not in problems:
                #     outFile.write(i2e[int(s)] + '\t' + i2r[int(p)] + '\t' + i2e[int(o)] + '\n')
                # else:
                #     cnt+=1
                outFile.write(i2e[int(s)] + '\t' + i2r[int(p)] + '\t' + i2e[int(o)] + '\n')

# print(cnt)
print(tot)

492672


In [40]:
# (shortened) entity types
typesdf = pd.read_csv(path+'entity_type.csv')

ent_df = pd.read_csv(path+'entities.txt',sep=r'\s+',
    header=None,
    names=['id', 'entity'],
    usecols=['entity']
)
merged_df = pd.merge(ent_df, typesdf, on='entity', how='inner')

merged_df['ntriple'] = (
    '<https://w3id.org/cskg/ontology#' + merged_df['entity'] +
    '>\t<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>\t<' +
    'https://w3id.org/cskg/ontology#' + merged_df['type']+'>\t.'
)

merged_df['ntriple'].to_csv(
    datapath + 'entity_types.nt',
    index=False,
    header=False,
    quoting=csv.QUOTE_NONE,
    escapechar=' '
)
                    # outFile.write(f'https://w3id.org/cskg/ontology#{row.entity}\t<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>\thttps://w3id.org/cskg/ontology#{row.type}\n')

In [8]:
df = pd.read_csv(datapath + 'entity_types.nt',sep='\t',header=None,names=['s','p','o','punkt'],usecols=['o'])
cskgclasses = list(df.o.unique().flatten())


In [13]:
from itertools import combinations

tboxgraph = Graph()
tboxgraph.parse(path+'cskg_schema.ttl', format='ttl')
for cls in cskgclasses:
    tboxgraph.add((URIRef(cls.strip('>').strip('<')), RDF.type, OWL.Class))

for class_a, class_b in combinations(cskgclasses, 2):
    tboxgraph.add((URIRef(class_a.strip('>').strip('<')), OWL.disjointWith, URIRef(class_b.strip('>').strip('<'))))

tboxgraph.parse(datapath+'CSKG2_entity_types.nt', format='nt')
tboxgraph.serialize(datapath+'CSKG2_tbox.nt', format="nt")
for fname in ['CSKG2_train.tsv', 'CSKG2_valid.tsv', 'CSKG2_test.tsv']:
    with open(datapath+fname) as fp:
        for line in fp:
            s, p, o = line.split()
            tboxgraph.add((URIRef(s),URIRef(p),URIRef(o)))
tboxgraph.serialize(datapath+'CSKG2_full_graph.nt', format="nt")

/Users/anon/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N0f758c8146eb4b2c858f26fa266465b2 (<class 'rdflib.graph.Graph'>)>

# NELL995 splits

In [57]:
default_ns = 'http://ste-lod-crew.fr/nell/ontology/'
ns = Namespace(default_ns)
g = Graph()

with open('NELL995/data/original/NELLKG0.txt') as inFile:
    with open('NELL995/data/NELL995_full_graph.tsv', 'w') as outFile:
        for line in inFile:
            line=  line.replace('__','_')
            s,p,o = line.split()
            sClass, sName = s.split('_',1)
            oClass, oName = o.split('_',1)

            sClass = default_ns+sClass
            sName = default_ns+ s
            oClass = default_ns+oClass
            oName = default_ns+ o
            pName = default_ns+p

            #outFile.write(sClass+'_'+sName + '\t' + pName + '\t' + oClass+'_'+oName + '\n')
            outFile.write(default_ns + s + '\t' +default_ns + p + '\t' +default_ns + o + '\n')

            g.add((URIRef(sName), RDF.type, URIRef(sClass)))
            g.add((URIRef(oName), RDF.type, URIRef(oClass)))
            g.add((URIRef(sName), URIRef(pName), URIRef(oName)))

    g.serialize('../datasets/NELL995/data/NELL995_full_graph.nt', format='ntriples')



/Users/anon/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [58]:
tf = TriplesFactory.from_path('NELL995/data/NELL995_full_graph.tsv')
training, testing, validation = tf.split([.8, .1, .1],random_state=42)

In [59]:
pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('NELL995/data/NELL995_train.tsv', header=False, index=False, sep ='\t')
pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('NELL995/data/NELL995_test.tsv', header=False, index=False, sep ='\t')
pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('NELL995/data/NELL995_valid.tsv', header=False, index=False, sep ='\t')



In [60]:
types_dict = defaultdict(set)
nell = pd.read_csv('NELL995/data/NELL995_full_graph.tsv', sep='\t', header=None, names = ['s','p','o'])
for i,r in nell.iterrows():
    subject = r.s
    subject_type = subject.split('_')[0]
    obj = r.o
    object_type = obj.split('_')[0]
    types_dict[subject].add(subject_type)
    types_dict[obj].add(object_type)

with open('NELL995/data/NELL995_entity_types.nt', 'w') as f:
    for subject, types_set in types_dict.items():
        for t in types_set:
            f.write('<'+subject+'>' + '\t<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>\t<' + t + '>\t.\n')

## Hetionet

In [1]:
base_iri = 'http://example.org/'

hn = Graph()

defined_classes = set()
property_definitions = defaultdict(lambda: {'domains': set(), 'ranges': set()})
property_map = dict()
tbox = pd.read_csv('metaedges.tsv', sep='\t')
for i,row in tbox.iterrows():
    prop_line = row['metaedge']

    if '-' in prop_line:
        prop_name = row['metaedge'].replace(' ','').split('-')[1]
        domain_str, prop_str, range_str = [item.replace(' ','') for item in prop_line.split(' - ')]

    else :
        prop_name= row['metaedge'].replace(' ','').split('>')[1]
        domain_str, prop_str, range_str = [item.replace(' ','') for item in prop_line.split(' > ')]

    property_map[row['abbreviation']] = prop_name
    property_definitions[prop_str]['domains'].add(domain_str)
    property_definitions[prop_str]['ranges'].add(range_str) # Also collect ranges for completeness

    if domain_str not in defined_classes:
        hn.add((URIRef(base_iri + domain_str.replace(' ','')), RDF.type, RDFS.Class))
        defined_classes.add(domain_str)
    if range_str not in defined_classes:
        hn.add((URIRef(base_iri + range_str.replace(' ','')), RDF.type, RDFS.Class))
        defined_classes.add(range_str)

for class_name in defined_classes:
    hn.add((URIRef(base_iri + class_name), RDF.type, RDFS.Class))

for prop_str, defs in property_definitions.items():
    prop_uri = URIRef(base_iri + prop_str.replace(' ',''))
    hn.add((prop_uri, RDF.type, RDF.Property))

    domain_uris = [URIRef(base_iri +d.replace(' ','')) for d in defs['domains']]
    if len(domain_uris) == 1:
        # Simple case: only one domain
        hn.add((prop_uri, RDFS.domain, domain_uris[0]))
    else:
        # Complex case: create a union class for the domain
        union_bnode = BNode() # An anonymous class for the union
        hn.add((union_bnode, RDF.type, RDFS.Class))
        # Use rdflib's Collection to create the RDF list for owl:unionOf
        c = Collection(hn, BNode(), domain_uris)
        hn.add((union_bnode, OWL.unionOf, c.uri))
        hn.add((prop_uri, RDFS.domain, union_bnode))

    range_uris = [URIRef(base_iri + r.replace(' ','')) for r in defs['ranges']]
    if len(range_uris) == 1:
        hn.add((prop_uri, RDFS.range, range_uris[0]))
    else:
        union_bnode = BNode()
        hn.add((union_bnode, RDF.type, RDFS.Class))
        c = Collection(hn, BNode(), range_uris)
        hn.add((union_bnode, OWL.unionOf, c.uri))
        hn.add((prop_uri, RDFS.range, union_bnode))
class_uris = [URIRef(base_iri + c.replace(' ','')) for c in defined_classes]

for class1 in class_uris:
    for class2 in class_uris:
        if class1 != class2:
            hn.add((class1, OWL.disjointWith, class2))
print(len(hn))
hn.serialize(format='nt', destination='hetionet/hetionet_tbox.nt')


NameError: name 'Graph' is not defined

In [ ]:
with open('hetionet/data/original/entity_mapping.nt','r') as f:

    for line in f.readlines():
        names = line.replace('<','').replace('>','')
        s,p,o,_ = names.split(' ')
        hn.add((URIRef(base_iri + s), RDF.type ,URIRef(base_iri + o)))

print(len(hn))

In [ ]:
hetio = datasets.Hetionet(42)
new_hetio = datasets.Dataset().from_tf(tf=hetio.merged(), ratios=[0.98,.01,0.01])
np.savetxt('hetionet/data/original/hetio_train.tsv', new_hetio.training.label_triples(new_hetio.training.mapped_triples), delimiter='\t', fmt='%s')
np.savetxt('hetionet/data/original/hetio_test.tsv', new_hetio.testing.label_triples(new_hetio.testing.mapped_triples), delimiter='\t', fmt='%s')
np.savetxt('hetionet/data/original/hetio_validation.tsv', new_hetio.validation.label_triples(new_hetio.validation.mapped_triples), delimiter='\t', fmt='%s')

In [ ]:
# add individual relations from train

with open('hetio_train.tsv', 'r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s, p, o = line.strip().split('\t')
        hn.add((URIRef(base_iri + s.split('::')[-1]), URIRef(base_iri + property_map.get(p)),
                URIRef(base_iri + o.split('::')[-1])))


hn.serialize('hetionet/hetio_train_graph.nt', format='nt')



In [ ]:
# make nicer datasets
for df_name in ['hetio_train.tsv', 'hetio_validation.tsv', 'hetio_test.tsv']:
    pre_df = pd.read_csv(df_name, sep='\t', header=None, names=['s', 'p','o'])
    pre_df['nice_p'] = pre_df.p.apply(lambda x: base_iri +property_map.get(x))
    pre_df['nice_s'] = pre_df.s.apply(lambda x: base_iri +x.split('::')[-1])
    pre_df['nice_o'] = pre_df.o.apply(lambda x: base_iri +x.split('::')[-1])
    pre_df[['nice_s','nice_p','nice_o']].to_csv('hetionet/'+ df_name.replace('.','_nice.'), sep='\t', index=False, header=False)

In [ ]:
for f_name in ['hetio_train_graph.nt', 'hetio_full_graph.nt']:
    with open(f_name, 'r') as in_f:
        with open(f_name.replace('.','_noIRI.'), 'w') as out_f:
            for line in in_f:
                out_f.write(line.replace('<','').replace('>','') + '\n')